# Alternative No-Answer Thresholding

In [ ]:
%pip install -q evaluate

In [ ]:
import numpy as np
import pandas as pd

from datasets import Dataset, load_dataset
from evaluate import load as load_metric
from sklearn.metrics import classification_report, f1_score
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, Trainer, TrainingArguments
from transformers import set_seed

set_seed(42)

In [ ]:
import sys
import torch
import transformers

print("Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)

# Load validation data and the saved QA model

In [ ]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_validation = dataset["validation"].to_pandas()
langlst = ["ar", "ko", "te"]
df_validation_filtered = df_validation[df_validation["lang"].isin(langlst)]

model_source = "multilingual-qa"
tokenizer = AutoTokenizer.from_pretrained(
    model_source, use_fast=True, local_files_only=True
)
model = AutoModelForQuestionAnswering.from_pretrained(
    model_source, local_files_only=True
)
training_args = TrainingArguments(
    output_dir=model_source,
    fp16=True,
    per_device_eval_batch_size=8,
    seed=42,
    full_determinism=True,
    report_to="none",
)
trainer = Trainer(model=model, args=training_args)

# Compute the core model's spans and no-answer scores

In [ ]:
validation_examples = Dataset.from_pandas(
    df_validation_filtered, preserve_index=False
)

def prepare_validation(examples):
    tokens = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    example_numbers = tokens.pop("overflow_to_sample_mapping")
    tokens["example_index"] = example_numbers

    for i in range(len(tokens["input_ids"])):
        sequence_ids = tokens.sequence_ids(i)
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]

    return tokens

# One batch keeps example_index aligned with the full validation dataframe.
validation_features = validation_examples.map(
    prepare_validation,
    batched=True,
    batch_size=len(validation_examples),
    remove_columns=validation_examples.column_names,
)
model_inputs = validation_features.remove_columns(
    ["offset_mapping", "example_index"]
)
start_logits, end_logits = trainer.predict(model_inputs).predictions

n = len(validation_examples)
null_scores = np.full(n, np.inf)
span_scores = np.full(n, -np.inf)
best_answers = [""] * n
MAX_ANSWER_LENGTH = 30

for feature_index, feature in enumerate(validation_features):
    example_index = feature["example_index"]
    offsets = feature["offset_mapping"]
    starts = start_logits[feature_index]
    ends = end_logits[feature_index]

    cls_index = feature["input_ids"].index(tokenizer.cls_token_id)
    null_scores[example_index] = min(
        null_scores[example_index],
        float(starts[cls_index] + ends[cls_index]),
    )

    for start, start_offset in enumerate(offsets):
        if start_offset is None or start_offset[1] <= start_offset[0]:
            continue

        for end in range(
            start, min(start + MAX_ANSWER_LENGTH, len(offsets))
        ):
            end_offset = offsets[end]
            if end_offset is None:
                break
            if end_offset[1] <= end_offset[0]:
                continue

            score = float(starts[start] + ends[end])
            if score > span_scores[example_index]:
                span_scores[example_index] = score
                context = validation_examples[example_index]["context"]
                best_answers[example_index] = context[
                    start_offset[0]:end_offset[1]
                ]

no_answer_scores = null_scores - span_scores
scores = np.asarray(no_answer_scores)

# Threshold selection

In [ ]:
def select_threshold(scores, gold_answerable):
    thresholds = [scores.min() - 1] + list(np.unique(scores))
    best_threshold = thresholds[0]
    best_macro_f1 = -1

    for threshold in thresholds:
        predicted_answerable = scores <= threshold
        macro_f1 = f1_score(
            gold_answerable,
            predicted_answerable,
            labels=[False, True],
            average="macro",
            zero_division=0,
        )

        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_threshold = threshold

    return float(best_threshold)

# Compare shared and language-specific thresholds

In [ ]:
gold_answerable = df_validation_filtered["answerable"].to_numpy(dtype=bool)
languages = df_validation_filtered["lang"].to_numpy()

shared_threshold = select_threshold(scores, gold_answerable)
language_thresholds = {}

for language in langlst:
    language_rows = languages == language
    language_thresholds[language] = select_threshold(
        scores[language_rows], gold_answerable[language_rows]
    )

print("Shared threshold:", shared_threshold)
print("Language-specific thresholds:", language_thresholds)

shared_predicted_answers = [
    answer if score <= shared_threshold else ""
    for answer, score in zip(best_answers, scores)
]
language_predicted_answers = [
    answer if score <= language_thresholds[language] else ""
    for answer, score, language in zip(best_answers, scores, languages)
]

# Validation results

In [ ]:
squad_metric = load_metric("squad_v2")
results = []

for method, predicted_answers in [
    ("Shared threshold", shared_predicted_answers),
    ("Language-specific thresholds", language_predicted_answers),
]:
    evaluation = df_validation_filtered.copy()
    evaluation["prediction"] = predicted_answers
    evaluation["pred_answerable"] = evaluation["prediction"] != ""
    evaluation["shared_pred_answerable"] = np.asarray(shared_predicted_answers) != ""

    for language in ["overall", "ar", "ko", "te"]:
        group = evaluation if language == "overall" else evaluation[evaluation["lang"] == language]
        formatted_predictions = []
        gold = []

        for example_id, example in group.iterrows():
            formatted_predictions.append({
                "id": str(example_id),
                "prediction_text": example["prediction"],
                # Encode the decision already made, not a calibrated probability.
                "no_answer_probability": float(not example["pred_answerable"]),
            })
            gold.append({
                "id": str(example_id),
                "answers": {
                    "text": [example["answer"]] if example["answerable"] else [],
                    "answer_start": [int(example["answer_start"])] if example["answerable"] else [],
                },
            })

        squad_results = squad_metric.compute(
            references=gold, predictions=formatted_predictions
        )
        report = classification_report(
            group["answerable"], group["pred_answerable"],
            labels=[False, True], target_names=["unanswerable", "answerable"],
            output_dict=True, zero_division=0,
        )

        shared_correct = group["shared_pred_answerable"] == group["answerable"]
        method_correct = group["pred_answerable"] == group["answerable"]

        row = {
            "method": method,
            "language": language,
            "N": len(group),
            "SQuAD_v2_EM": squad_results["exact"],
            "SQuAD_v2_token_F1": squad_results["f1"],
            "answerability_macro_F1": 100 * report["macro avg"]["f1-score"],
            "wrong_to_correct": (~shared_correct & method_correct).sum(),
            "correct_to_wrong": (shared_correct & ~method_correct).sum(),
        }
        for label in ["unanswerable", "answerable"]:
            row[f"{label}_P"] = 100 * report[label]["precision"]
            row[f"{label}_R"] = 100 * report[label]["recall"]
            row[f"{label}_F1"] = 100 * report[label]["f1-score"]
        results.append(row)

# Store the shared-threshold score for each language and overall.
shared_macro_f1 = {}
for row in results:
    if row["method"] == "Shared threshold":
        language = row["language"]
        shared_macro_f1[language] = row["answerability_macro_F1"]

# Subtract the baseline score for the same language.
for row in results:
    language = row["language"]
    baseline_score = shared_macro_f1[language]
    row["macro_F1_change"] = row["answerability_macro_F1"] - baseline_score

results_table = pd.DataFrame(results)
display(results_table.round(2))